In [2]:
# --- STEP 0: System Setup ---
! git clone https://github.com/sreedharkaranam-ai/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

Cloning into 'edu-101-java-code'...
remote: Enumerating objects: 578, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 578 (delta 71), reused 67 (delta 53), pack-reused 468 (from 2)
Receiving objects: 100% (578/578), 30.23 MiB | 29.31 MiB/s, done.
Resolving deltas: 100% (227/227), done.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
cluster2.db  edu-101-java-code	pom.xml  src  target


In [3]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [5]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
import subprocess
import time

# 1. Start the Temporal server in the background (equivalent to trailing '&')
print("Starting Temporal server...")
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081','--db-filename','cluster2.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")

# Note: The server_process is still running in the background here.
# You can now run your Gradle command via subprocess, or keep the script running.

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
Starting Temporal server...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Tem

## --- EXAMPLE 2 ---


In [6]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8082','--db-filename','cluster3.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")
print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
19:10:28.476 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:10:30.622 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=25794@e0fc39f6d057} 
19:10:30.715 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=25794@e0fc39f6d057} 
Worker started for task queue: HelloAccumulatorTaskQue

In [5]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

18:15:12.910 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:15:13.686 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=8249@e0fc39f6d057} 
18:15:13.727 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=8249@e0fc39f6d057} 
Worker started for task queue: HelloAccumulatorTaskQueue
18:15:14.641 {HelloAccumulatorWorkflow-yellow } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Missy Robot, bucket=yellow, greetingKey=key-0] 
18:15:14.642 {HelloAccumulatorWorkflow-blue } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - rec

In [6]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


18:17:22.476 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:17:23.423 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=8876@e0fc39f6d057} 
18:17:23.482 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=8876@e0fc39f6d057} 
18:17:24.921 {HelloActivityWorkflow b1f88322-8d90-3e1c-aeee-acd90cd2f05b} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


18:17:33.140 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:17:33.944 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=9010@e0fc39f6d057} 
18:17:33.983 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=9010@e0fc39f6d057} 
composeGreeting activity is going to fail
18:17:34.904 {HelloActivityWithRetriesWorkflow 33938f50-8121-3094-b999-393bdeb61375} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=33938f50-8121-3094-b999-393bdeb61375, activityType=ComposeGreeting, attempt=1 
j

In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


18:17:51.195 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:17:52.229 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=9173@e0fc39f6d057} 
18:17:52.269 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=9173@e0fc39f6d057} 
Order results: Ordered 8 Apples...Ordered 5 Bananas...Ordered 1 Cherries...Ordered 4 Oranges...


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


18:17:58.728 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:17:59.485 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=9303@e0fc39f6d057} 
18:17:59.526 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=9303@e0fc39f6d057} 
Hello World!
Bye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


18:18:10.124 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:11.313 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=9430@e0fc39f6d057} 
18:18:11.374 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=9430@e0fc39f6d057} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


18:18:17.750 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:18.641 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=9570@e0fc39f6d057} 
18:18:18.688 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=9570@e0fc39f6d057} 
Hello World!


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


18:18:28.023 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:29.552 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=9696@e0fc39f6d057} 
18:18:29.619 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=9696@e0fc39f6d057} 
Hello World!
Hello World!


In [13]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


18:18:36.878 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:37.673 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=9841@e0fc39f6d057} 
18:18:37.705 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=9841@e0fc39f6d057} 
18:18:40.494 {HelloDetachedCancellationWorkflow f501c080-fba8-369e-ac59-088fec5f0cf4} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=f501c080-fba8-369e-ac59-088fec5f0cf4, activityType=SayHello, attempt=1 
Goodbye John!


In [14]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


18:18:49.488 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:50.639 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=9982@e0fc39f6d057} 
Hello World!


In [15]:
!timeout 240s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

18:18:57.373 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:18:58.176 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=10108@e0fc39f6d057} 
18:18:58.219 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=10108@e0fc39f6d057} 
Started workflow_id: "HelloCronWorkflow"
run_id: "019cd3d2-e076-7899-ad04-b3d2d471cbba"

From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!


In [16]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


18:22:57.903 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:22:58.675 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=11171@e0fc39f6d057} 
18:22:58.722 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=11171@e0fc39f6d057} 
DynamicACT: Hello John from: DynamicWF


In [17]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


18:23:09.295 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:23:10.210 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=11301@e0fc39f6d057} 
18:23:10.259 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=11301@e0fc39f6d057} 
18:23:11.186 {HelloEagerWorkflowStartWorkflow 720ea2e2-dadf-3494-a7b2-11009a1b29ab} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [18]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


18:23:16.528 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:23:17.407 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=11431@e0fc39f6d057} 
18:23:17.447 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=11431@e0fc39f6d057} 
18:23:19.052 {be78a17c-ca49-39f6-9535-6c6e6215248d b513a5a0-9131-36e9-bd98-cc20fa715899} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=b513a5a0-9131-36e9-bd98-cc20fa715899, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [19]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


18:23:29.630 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:23:30.372 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=11571@e0fc39f6d057} 
18:23:30.397 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=11571@e0fc39f6d057} 
Hello World!


In [20]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


18:23:36.305 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:23:37.126 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=11702@e0fc39f6d057} 
18:23:37.166 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=11702@e0fc39f6d057} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5334 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4979 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4436 milliseconds and then I will greet you agai

In [21]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


18:24:04.521 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:24:05.361 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=11909@e0fc39f6d057} 
18:24:05.403 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=11909@e0fc39f6d057} 
Hello World!
Bye World!



In [22]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


18:24:11.556 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:24:12.286 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=12031@e0fc39f6d057} 
Hello World!
Bye World!


In [23]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


18:24:23.231 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:24:23.993 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=12162@e0fc39f6d057} 
18:24:24.024 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=12162@e0fc39f6d057} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [24]:
!timeout 300s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules

18:24:32.378 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:24:33.749 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=12292@e0fc39f6d057} 
18:24:33.806 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=12292@e0fc39f6d057} 
From HelloScheduleWorkflow-2026-03-09T18:24:34Z: Hello World from HelloSchedule scheduled at 2026-03-09T18:24:34Z!
From HelloScheduleWorkflow-2026-03-09T18:24:40Z: Hello World from HelloSchedule scheduled at 2026-03-09T18:24:40Z!
From HelloScheduleWorkflow-2026-03-09T18:24:45Z: Hello World from HelloSchedule scheduled at 2026-03-09T18:24:45Z!
From HelloScheduleWorkflow-2026-03-09T18:24:50Z: Hello World

In [25]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


18:25:33.158 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
18:25:33.872 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=12630@e0fc39f6d057} 
[Hello World!, Hello Universe!]


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes

19:13:16.790 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:13:17.720 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=26590@e0fc39f6d057} 
19:13:17.749 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=26590@e0fc39f6d057} 
In workflow we get CustomKeywordField is: keys
Hello SearchAttributes!


In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


19:13:32.396 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:13:33.191 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=26744@e0fc39f6d057} 
19:13:33.245 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=26744@e0fc39f6d057} 
Hello TypedSearchAttributes how are you doing?


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



19:13:49.752 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:13:50.692 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=26904@e0fc39f6d057} 
19:13:50.746 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=26904@e0fc39f6d057} 
Goodbye World!
Goodbye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


19:14:00.065 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:14:01.756 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=27046@e0fc39f6d057} 
19:14:01.817 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=27046@e0fc39f6d057} 
19:14:03.352 {HelloUpdateWorkflow 60c4e1d8-32f3-3a2e-988a-f1f3e6bfc8c2} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
19:14:03.628 {HelloUpdateWorkflow 9ff21b7e-e664-39d3-b6c5-7ee997910ee9} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


19:17:21.337 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:17:22.284 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=27944@e0fc39f6d057} 
19:17:22.326 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=27944@e0fc39f6d057} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 8
Processing value: Value 10
19:17:48.219 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-6c8c3e65-5865-441f-aacb-ad2e74ced9f4] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

19:17:59.729 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
19:18:00.676 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=28194@e0fc39f6d057} 
19:18:00.703 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=28194@e0fc39f6d057} 
Result: Hello Michael Jordan,Hello John Stockton
19:18:02.561 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=a926ffcd-6a6a-45b2-8f83-15af19085b0d, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [32]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
